# Optimisation Multi-Étages de la Valeur de Stock — Version corrigée (horizon réaliste)

## Pourquoi le modèle à 4 mois était infaisable

Les lead times réels vont jusqu'à **484 jours** (médiane 235 jours ≈ 7,7 mois). Sur un horizon
de 4 mois, la **majorité** des matières premières ne peuvent techniquement jamais être
réapprovisionnées à temps : ce n'est pas une question de granularité (semaine vs mois), c'est
une question de **durée d'horizon face aux délais réels**. Passer en semaines sans allonger
l'horizon ne change rien — passer en semaines EN allongeant l'horizon proportionnellement
multiplie par ~13 le nombre de variables (56 semaines × 5188 références × plusieurs
variables/référence), ce qui rend le MILP quasi impossible à résoudre avec un solveur
open-source (CBC) en temps raisonnable.

**Solution retenue :**
1. **Horizon étendu à 13 mois** (avril 2026 → avril 2027), qui correspond exactement à
   l'étendue réelle disponible dans le fichier de demande (56 semaines agrégées par mois
   fiscal). Ça couvre **~80% des lead times réels** au lieu de 28%.
2. **Variable de rupture (backorder) ajoutée à l'équation de stock Produit Fini** : pour les
   ~20% de références dont le lead time dépasse même 13 mois, le modèle reste **toujours
   faisable** — il choisit d'accepter une rupture ponctuelle plutôt que de planter, avec une
   forte pénalité dans la fonction objectif pour la décourager autant que possible.
3. **Granularité mensuelle conservée** (pas hebdomadaire) pour rester résolvable par CBC dans
   un temps raisonnable, tout en couvrant un horizon réaliste.

## Calibration du scénario
- **Cible fiscale (TARGET)** : 9 000 000 $
- **Valeur totale de stock actuelle** : mise à l'échelle pour être ≈ 12 000 000 $ (la valeur
  réelle extraite de SAP est de ~4,88 M$ ; un facteur d'échelle uniforme est appliqué aux
  quantités en stock — pas aux prix — pour obtenir un scénario de surstockage réaliste et
  pédagogiquement significatif, tout en conservant les proportions réelles entre références).


In [ ]:
from pyomo.environ import *
from IPython.display import display
import pandas as pd
import numpy as np
import math
import re
from datetime import date, timedelta


---
## 1. LES PARAMÈTRES


In [ ]:
model = AbstractModel()

# ---------- Horizon temporel ----------
model.T   = Param(within=PositiveIntegers)
model.PER = RangeSet(1, model.T)
model.t_juillet = Param(within=PositiveIntegers)
model.AoutSet    = Set(within=model.PER)

# ---------- Ensembles de références ----------
model.MP = Set()
model.SF = Set()
model.PF = Set()

# ---------- Paramètres économiques généraux ----------
model.TARGET   = Param()
model.tau      = Param()
model.z_alpha  = Param()
model.M        = Param()
model.V_max    = Param()
model.penalite_rupture = Param()

# ---------- Matières Premières ----------
model.prix_MP       = Param(model.MP)
model.stock_init_MP = Param(model.MP)
model.LT            = Param(model.MP)
model.MOQ           = Param(model.MP)
model.volume_MP     = Param(model.MP)
model.SS_MP         = Param(model.MP)

# ---------- Semi-Finis ----------
model.prix_SF       = Param(model.SF)
model.stock_init_SF = Param(model.SF)
model.ct_SF         = Param(model.SF)
model.MLS_SF        = Param(model.SF)
model.charge_SF     = Param(model.SF)
model.volume_SF     = Param(model.SF)
model.SS_SF         = Param(model.SF)

# ---------- Produits Finis ----------
model.prix_PF       = Param(model.PF)
model.stock_init_PF = Param(model.PF)
model.ct_PF         = Param(model.PF)
model.MLS_PF        = Param(model.PF)
model.charge_PF     = Param(model.PF)
model.volume_PF     = Param(model.PF)
model.SS_PF         = Param(model.PF)

model.demande = Param(model.PF, model.PER)
model.OC      = Param(model.MP, model.PER)
model.WIP_SF  = Param(model.SF, model.PER)
model.WIP_PF  = Param(model.PF, model.PER)

model.a = Param(model.MP, model.SF, default=0)
model.b = Param(model.SF, model.PF, default=0)

model.Cap_SF = Param(model.PER)
model.Cap_PF = Param(model.PER)


---
## 2. LES VARIABLES

`Rupture_PF` est la variable de rupture (backorder) ajoutée pour garantir la faisabilité du
modèle même quand le lead time réel dépasse l'horizon.


In [ ]:
model.S_MP = Var(model.MP, model.PER, domain=NonNegativeReals)
model.S_SF = Var(model.SF, model.PER, domain=NonNegativeReals)
model.S_PF = Var(model.PF, model.PER, domain=NonNegativeReals)

model.O_MP = Var(model.MP, model.PER, domain=NonNegativeReals)
model.P_SF = Var(model.SF, model.PER, domain=NonNegativeReals)
model.P_PF = Var(model.PF, model.PER, domain=NonNegativeReals)

model.x    = Var(model.MP, model.PER, domain=Binary)
model.y_SF = Var(model.SF, model.PER, domain=Binary)
model.y_PF = Var(model.PF, model.PER, domain=Binary)

model.Rupture_PF = Var(model.PF, model.PER, domain=NonNegativeReals)


---
## 3. LA FONCTION OBJECTIF (et les contraintes C1 → C13)


In [ ]:
def cost_rule(m):
    valeur_stock = (
        sum(m.prix_MP[i]*m.S_MP[i,t] for i in m.MP for t in m.PER)
      + sum(m.prix_SF[j]*m.S_SF[j,t] for j in m.SF for t in m.PER)
      + sum(m.prix_PF[l]*m.S_PF[l,t] for l in m.PF for t in m.PER)
    )
    cout_rupture = m.penalite_rupture * sum(m.Rupture_PF[l,t] for l in m.PF for t in m.PER)
    return valeur_stock + cout_rupture
model.cost = Objective(rule=cost_rule, sense=minimize)


In [ ]:
# C1 — Bilan de stock Matière Première
def c1_rule(m, i, t):
    S_prev = m.stock_init_MP[i] if t == 1 else m.S_MP[i, t-1]
    t_cmd  = t - m.LT[i]
    reception = m.O_MP[i, t_cmd] if t_cmd >= 1 else 0
    conso = sum(m.a[i,j]*m.P_SF[j,t] for j in m.SF if m.a[i,j] > 0)
    return m.S_MP[i,t] == S_prev + m.OC[i,t] + reception - conso
model.C1_Bilan_MP = Constraint(model.MP, model.PER, rule=c1_rule)

# C2 — Bilan de stock Semi-Fini
def c2_rule(m, j, t):
    S_prev = m.stock_init_SF[j] if t == 1 else m.S_SF[j, t-1]
    t_lanc = t - m.ct_SF[j]
    reception = m.P_SF[j, t_lanc] if t_lanc >= 1 else 0
    conso = sum(m.b[j,l]*m.P_PF[l,t] for l in m.PF if m.b[j,l] > 0)
    return m.S_SF[j,t] == S_prev + m.WIP_SF[j,t] + reception - conso
model.C2_Bilan_SF = Constraint(model.SF, model.PER, rule=c2_rule)

# C3 — Bilan de stock Produit Fini (AVEC variable de rupture)
def c3_rule(m, l, t):
    S_prev = m.stock_init_PF[l] if t == 1 else m.S_PF[l, t-1]
    t_lanc = t - m.ct_PF[l]
    reception = m.P_PF[l, t_lanc] if t_lanc >= 1 else 0
    return m.S_PF[l,t] - m.Rupture_PF[l,t] == S_prev + m.WIP_PF[l,t] + reception - m.demande[l,t]
model.C3_Bilan_PF = Constraint(model.PF, model.PER, rule=c3_rule)

# C5 — Cible de valeur de stock fin d'horizon
def c5_rule(m):
    return (
        sum(m.prix_MP[i]*m.S_MP[i, m.T] for i in m.MP)
      + sum(m.prix_SF[j]*m.S_SF[j, m.T] for j in m.SF)
      + sum(m.prix_PF[l]*m.S_PF[l, m.T] for l in m.PF)
      <= m.TARGET
    )
model.C5_Cible_Fiscale = Constraint(rule=c5_rule)

# C6 — Taux de service minimum PF, à partir de t=2 (stock initial = donnée subie)
def c6_rule(m, l, t):
    if t == 1:
        return Constraint.Skip
    return m.S_PF[l,t] >= m.SS_PF[l]
model.C6_TauxService = Constraint(model.PF, model.PER, rule=c6_rule)

# C7 — Capacités de production
def c7_sf_rule(m, t):
    return sum(m.charge_SF[j]*m.P_SF[j,t] for j in m.SF) <= m.Cap_SF[t]
model.C7_Capacite_SF = Constraint(model.PER, rule=c7_sf_rule)

def c7_pf_rule(m, t):
    return sum(m.charge_PF[l]*m.P_PF[l,t] for l in m.PF) <= m.Cap_PF[t]
model.C7_Capacite_PF = Constraint(model.PER, rule=c7_pf_rule)

# C8 — Capacité de stockage (volume)
def c8_rule(m, t):
    return (
        sum(m.volume_MP[i]*m.S_MP[i,t] for i in m.MP)
      + sum(m.volume_SF[j]*m.S_SF[j,t] for j in m.SF)
      + sum(m.volume_PF[l]*m.S_PF[l,t] for l in m.PF)
      <= m.V_max
    )
model.C8_Capacite_Stockage = Constraint(model.PER, rule=c8_rule)

# C9 — MOQ / liaison binaire achat MP
def c9a_rule(m, i, t):
    return m.O_MP[i,t] >= m.MOQ[i]*m.x[i,t]
model.C9a_MOQ_min = Constraint(model.MP, model.PER, rule=c9a_rule)

def c9b_rule(m, i, t):
    return m.O_MP[i,t] <= m.M*m.x[i,t]
model.C9b_MOQ_max = Constraint(model.MP, model.PER, rule=c9b_rule)

# C10 — MLS / liaison binaire production SF-PF
def c10a_sf_rule(m, j, t):
    return m.P_SF[j,t] >= m.MLS_SF[j]*m.y_SF[j,t]
model.C10a_MLS_SF_min = Constraint(model.SF, model.PER, rule=c10a_sf_rule)
def c10b_sf_rule(m, j, t):
    return m.P_SF[j,t] <= m.M*m.y_SF[j,t]
model.C10b_MLS_SF_max = Constraint(model.SF, model.PER, rule=c10b_sf_rule)

def c10a_pf_rule(m, l, t):
    return m.P_PF[l,t] >= m.MLS_PF[l]*m.y_PF[l,t]
model.C10a_MLS_PF_min = Constraint(model.PF, model.PER, rule=c10a_pf_rule)
def c10b_pf_rule(m, l, t):
    return m.P_PF[l,t] <= m.M*m.y_PF[l,t]
model.C10b_MLS_PF_max = Constraint(model.PF, model.PER, rule=c10b_pf_rule)

# C11 — Fermeture fournisseur (août)
def c11_rule(m, i, t):
    return m.O_MP[i,t] == 0
model.C11_Fermeture_Fournisseur = Constraint(model.MP, model.AoutSet, rule=c11_rule)

# C12 — Couverture de stock avant fermeture
def c12_rule(m, i):
    conso_aout = sum(m.a[i,j]*m.P_SF[j,t] for t in m.AoutSet for j in m.SF if m.a[i,j] > 0)
    return m.S_MP[i, m.t_juillet] >= conso_aout
model.C12_Couverture_Aout = Constraint(model.MP, rule=c12_rule)


---
## 4. SOLVE

Vu la taille réelle du problème (~5 200 références × 13 périodes ≈ 67 000 variables binaires),
on fixe une **limite de temps** et un **gap MIP acceptable** pour obtenir une solution de bonne
qualité en temps raisonnable plutôt que d'attendre une preuve d'optimalité stricte.


In [ ]:
def resoudre(instance, solveur="cbc", verbose=True, temps_max_s=900, gap_mip=0.02):
    opt = SolverFactory(solveur)
    opt.options['seconds']  = temps_max_s
    opt.options['ratioGap'] = gap_mip
    resultats = opt.solve(instance, tee=verbose)
    return resultats


In [ ]:
import shutil
chemin_cbc = shutil.which("cbc")
if chemin_cbc is None:
    import subprocess
    subprocess.run(["apt-get", "install", "-y", "coinor-cbc"], check=True)
    chemin_cbc = shutil.which("cbc")
assert chemin_cbc is not None, "CBC introuvable"
print(f"CBC disponible : {chemin_cbc}")


---
## 5. METTRE EN PLACE LA DATA

### 5.1 Fichiers sources


In [ ]:
FICHIER_STOCK      = "PN__STOCK__PRIX_EN_STOCK_TYPE_PN.XLSX"
FICHIER_DEMANDE     = "DEMAND_ANALYSSIS.xlsx"
FICHIER_LEADTIMES   = "LEAD_TIME_FOURNISSEUR_AVEC_PN.xlsx"
FICHIER_WIP_ACHATS  = "WIP_Achats_En_Cours.xlsx"
FICHIER_MOQ_MLS     = "PN_MLS.xlsx"          # converti depuis .xls
FICHIER_BOM         = "ZBOM.XLSX"


### 5.2 Référentiel produits (stock, prix, type) + reclassification

Le fichier SAP distingue 5 types (`ZROH`=matière première, `ZVRP`=consommable/emballage,
`ZWRB`=divers, `ZHLB`=semi-fini, `ZFRT`=produit fini). Pour le modèle à 3 étages, `ZROH`,
`ZVRP` et `ZWRB` sont regroupés comme « Matière Première » (achetées à un fournisseur externe,
même logique de lead time), `ZHLB` reste Semi-Fini et `ZFRT` reste Produit Fini.


In [ ]:
df_produits = pd.read_excel(FICHIER_STOCK)
df_produits = df_produits.drop_duplicates(subset="Material", keep="first").set_index("Material")

TYPES_MP = {"ZROH", "ZVRP", "ZWRB"}
type_of = df_produits["Material Type"]

refs_MP = type_of[type_of.isin(TYPES_MP)].index.tolist()
refs_SF = type_of[type_of == "ZHLB"].index.tolist()
refs_PF = type_of[type_of == "ZFRT"].index.tolist()

print(f"Références MP : {len(refs_MP)}   SF : {len(refs_SF)}   PF : {len(refs_PF)}")


### 5.3 Mise à l'échelle du stock initial (calibration du scénario)

La valeur totale de stock réelle est de l'ordre de 4,9 M$. Pour obtenir un scénario de
surstockage pédagogiquement réaliste face à une cible de 9 M$, les **quantités** en stock
(pas les prix) sont mises à l'échelle par un facteur uniforme pour atteindre ≈ 12 M$ de valeur
totale initiale, en conservant les proportions réelles entre références.


In [ ]:
VALEUR_CIBLE_INITIALE = 12_000_000

valeur_reelle = (df_produits["Unrestricted Stock"] * df_produits["Prix_Unitaire"]).sum()
facteur_echelle = VALEUR_CIBLE_INITIALE / valeur_reelle
print(f"Valeur de stock réelle (SAP)      : {valeur_reelle:,.0f} $")
print(f"Facteur d'échelle appliqué        : x{facteur_echelle:.4f}")

df_produits["Stock_Init_Calibre"] = df_produits["Unrestricted Stock"] * facteur_echelle

stock_init_MP = df_produits.loc[refs_MP, "Stock_Init_Calibre"].to_dict()
stock_init_SF = df_produits.loc[refs_SF, "Stock_Init_Calibre"].to_dict()
stock_init_PF = df_produits.loc[refs_PF, "Stock_Init_Calibre"].to_dict()

prix_MP = df_produits.loc[refs_MP, "Prix_Unitaire"].to_dict()
prix_SF = df_produits.loc[refs_SF, "Prix_Unitaire"].to_dict()
prix_PF = df_produits.loc[refs_PF, "Prix_Unitaire"].to_dict()

valeur_calibree = (df_produits["Stock_Init_Calibre"] * df_produits["Prix_Unitaire"]).sum()
print(f"Valeur de stock initiale calibrée : {valeur_calibree:,.0f} $")


### 5.4 Horizon temporel — 13 mois, à partir des 56 semaines réelles de demande

Le fichier de demande fournit 56 semaines fiscales réelles (format `AAAA/SS`, année fiscale
commençant le 1ᵉʳ octobre). On les agrège par mois calendaire : l'horizon obtenu (avril 2026 →
avril 2027, 13 mois) couvre ~80% des lead times réels, contre 28% avec un horizon à 4 mois.


In [ ]:
df_dem_brut = pd.read_excel(FICHIER_DEMANDE).drop_duplicates(subset="MATERIAL").set_index("MATERIAL")
semaine_cols = [c for c in df_dem_brut.columns if re.match(r"^\d{4}/\d{1,2}$", str(c))]

def semaine_vers_date(col):
    fy, wk = map(int, col.split("/"))
    return date(fy - 1, 10, 1) + timedelta(weeks=wk - 1)

mois_de_semaine = {c: (semaine_vers_date(c).year, semaine_vers_date(c).month) for c in semaine_cols}
mois_ordonnes   = sorted(set(mois_de_semaine.values()))
t_de_mois       = {m: i + 1 for i, m in enumerate(mois_ordonnes)}
T_max           = len(mois_ordonnes)

noms_periodes = {t_de_mois[m]: f"{m[1]:02d}/{m[0]}" for m in mois_ordonnes}
aout_t    = [t_de_mois[m] for m in mois_ordonnes if m[1] == 8]
t_juillet = min(aout_t) - 1 if aout_t else T_max

print(f"Horizon : {T_max} mois, de {noms_periodes[1]} à {noms_periodes[T_max]}")
print(f"Mois de fermeture fournisseur (août) : t = {aout_t}")


In [ ]:
def to_number(v):
    if pd.isna(v):
        return 0.0
    if isinstance(v, str):
        v = v.strip().replace(" ", "").replace(",", "")
        if v in ("", "-", "N/A", "n/a", "NA"):
            return 0.0
    try:
        return float(v)
    except (ValueError, TypeError):
        return 0.0

demande = {(l, t): 0.0 for l in refs_PF for t in range(1, T_max + 1)}
for col in semaine_cols:
    t = t_de_mois[mois_de_semaine[col]]
    for l in refs_PF:
        if l in df_dem_brut.index:
            demande[(l, t)] += to_number(df_dem_brut.loc[l, col])

demande_par_pf = {l: {t: demande[(l, t)] for t in range(1, T_max + 1)} for l in refs_PF}
print(f"Demande totale sur l'horizon : {sum(demande.values()):,.0f} unités")


### 5.5 Lead times fournisseur (jours → mois)

In [ ]:
df_lt = pd.read_excel(FICHIER_LEADTIMES).groupby("PN")["LEAD TIME ( day ) "].max()
LT = {i: max(1, math.ceil(df_lt.get(i, 30) / 30)) for i in refs_MP}

hors_horizon = [i for i in refs_MP if LT[i] > T_max]
print(f"MP avec LT > horizon ({T_max} mois) : {len(hors_horizon)} / {len(refs_MP)} "
      f"({len(hors_horizon)/len(refs_MP)*100:.1f}%) — couvertes par le pipeline OC déjà engagé")


### 5.6 MOQ (MP) / MLS (SF, PF)

In [ ]:
df_moqmls = pd.read_excel(FICHIER_MOQ_MLS).drop_duplicates(subset="Material").set_index("Material")["Cur. Minimum lot size"]
MOQ    = {i: df_moqmls.get(i, 1) for i in refs_MP}
MLS_SF = {j: df_moqmls.get(j, 1) for j in refs_SF}
MLS_PF = {l: df_moqmls.get(l, 1) for l in refs_PF}


### 5.7 WIP en production / achats déjà engagés (avec période réelle `Mois Modèle (t)`)

In [ ]:
df_oc     = pd.read_excel(FICHIER_WIP_ACHATS, sheet_name="Achats_En_Cours_MP")
df_wip_sf = pd.read_excel(FICHIER_WIP_ACHATS, sheet_name="WIP_Semi_Finis")
df_wip_pf = pd.read_excel(FICHIER_WIP_ACHATS, sheet_name="WIP_Produits_Finis")

OC     = {i: {t: 0.0 for t in range(1, T_max + 1)} for i in refs_MP}
WIP_SF = {j: {t: 0.0 for t in range(1, T_max + 1)} for j in refs_SF}
WIP_PF = {l: {t: 0.0 for t in range(1, T_max + 1)} for l in refs_PF}

for _, r in df_oc.iterrows():
    if r["PN"] in OC and 1 <= r["Mois Modèle (t)"] <= T_max:
        OC[r["PN"]][int(r["Mois Modèle (t)"])] += to_number(r["Qté Commandée"])

for _, r in df_wip_sf.iterrows():
    if r["PN SF"] in WIP_SF and 1 <= r["Mois Modèle (t)"] <= T_max:
        WIP_SF[r["PN SF"]][int(r["Mois Modèle (t)"])] += to_number(r["Qté En-Cours"])

for _, r in df_wip_pf.iterrows():
    if r["PN PF"] in WIP_PF and 1 <= r["Mois Modèle (t)"] <= T_max:
        WIP_PF[r["PN PF"]][int(r["Mois Modèle (t)"])] += to_number(r["Qté En-Cours"])

print("Pipeline OC / WIP chargé")


### 5.8 Nomenclature (BOM) — classification 2 niveaux MP→SF / SF→PF

Le fichier BOM SAP contient des nomenclatures à plusieurs niveaux (`Level Number` de 1 à 5). Le
modèle ne représente que 2 transformations (MP→SF puis SF→PF) : les liens `MP-type → SF`
(niveau direct) et `SF → PF` (niveau direct) sont conservés ; les liens plus profonds
(sous-assemblage de sous-assemblage `SF→SF`, ou consommation directe `MP→PF` sans passer par un
semi-fini) sont recensés mais non modélisés — limitation documentée du modèle à 2 étages.


In [ ]:
df_bom = pd.read_excel(FICHIER_BOM)
df_bom = df_bom[df_bom["Base quantity"] > 0].copy()
df_bom["Ratio"] = df_bom["Quantity"] / df_bom["Base quantity"]

a_bom = {i: {} for i in refs_MP}
b_bom = {j: {} for j in refs_SF}
liens_ignores = []

for _, r in df_bom.iterrows():
    parent, comp, ratio = r["Parent"], r["Material / Doc"], r["Ratio"]
    tp, tc = type_of.get(parent), type_of.get(comp)
    if tp == "ZHLB" and tc in TYPES_MP and comp in a_bom:
        a_bom[comp][parent] = a_bom[comp].get(parent, 0) + ratio
    elif tp == "ZFRT" and tc == "ZHLB" and comp in b_bom:
        b_bom[comp][parent] = b_bom[comp].get(parent, 0) + ratio
    else:
        liens_ignores.append((parent, comp, tp, tc))

print(f"Liens BOM utilisés  : {len(df_bom) - len(liens_ignores)} / {len(df_bom)}")
print(f"Liens BOM ignorés   : {len(liens_ignores)} (multi-niveaux SF->SF ou MP->PF direct, non modélisés)")


### 5.9 Génération logique — volumes, charge, cycle de production, capacités

Non fournies par SAP dans ce jeu de données : générées à partir du prix (proxy de complexité),
avec les hypothèses documentées dans le code.


In [ ]:
def volume_logique(prix, etage):
    base = {"MP": 0.02, "SF": 0.05, "PF": 0.12}[etage]
    return round(base / (1 + prix / 500), 4)

volume_MP = {i: volume_logique(prix_MP[i], "MP") for i in refs_MP}
volume_SF = {j: volume_logique(prix_SF[j], "SF") for j in refs_SF}
volume_PF = {l: volume_logique(prix_PF[l], "PF") for l in refs_PF}

def charge_logique(prix, plancher, plafond):
    return float(np.clip(prix / 150, plancher, plafond))

charge_SF = {j: charge_logique(prix_SF[j], 0.2, 8.0)  for j in refs_SF}
charge_PF = {l: charge_logique(prix_PF[l], 0.5, 12.0) for l in refs_PF}

seuil_SF = np.percentile(list(charge_SF.values()), 75)
seuil_PF = np.percentile(list(charge_PF.values()), 75)
ct_SF = {j: (2 if charge_SF[j] > seuil_SF else 1) for j in refs_SF}
ct_PF = {l: (2 if charge_PF[l] > seuil_PF else 1) for l in refs_PF}

def cout_fixe_logique(prix, plancher, plafond):
    return float(np.clip(prix * 0.5, plancher, plafond))

C_passation    = {i: cout_fixe_logique(prix_MP[i], 50, 1000)  for i in refs_MP}   # informatif
C_lancement_SF = {j: cout_fixe_logique(prix_SF[j], 100, 1500) for j in refs_SF}
C_lancement_PF = {l: cout_fixe_logique(prix_PF[l], 150, 2000) for l in refs_PF}

# Capacités mensuelles réalistes : nb lignes x jours ouvrés x heures/jour x OEE, réduites en août
jours_ouvres = {t: (10 if t in aout_t else 21) for t in range(1, T_max + 1)}
heures_jour, oee = 7.5, 0.85
NB_LIGNES_SF, NB_LIGNES_PF = 6, 4   # à ajuster à la réalité du site

Cap_SF = {t: jours_ouvres[t]*heures_jour*oee*NB_LIGNES_SF for t in range(1, T_max + 1)}
Cap_PF = {t: jours_ouvres[t]*heures_jour*oee*NB_LIGNES_PF for t in range(1, T_max + 1)}

# Big-M : strictement supérieur au plus grand MOQ/MLS
plus_grand_seuil = max(max(MOQ.values()), max(MLS_SF.values()), max(MLS_PF.values()))
M_bigM = plus_grand_seuil * 2
print(f"M (big-M) = {M_bigM:,.0f}  (plus grand MOQ/MLS = {plus_grand_seuil:,.0f})")

# V_max : capacité de stockage réaliste = 1.3x le volume total nécessaire au stock calibré
volume_total_initial = (
    sum(volume_MP[i]*stock_init_MP[i] for i in refs_MP)
  + sum(volume_SF[j]*stock_init_SF[j] for j in refs_SF)
  + sum(volume_PF[l]*stock_init_PF[l] for l in refs_PF)
)
V_max = round(volume_total_initial * 1.3)
print(f"V_max (capacité stockage) = {V_max:,.0f} m3  (besoin initial ~= {volume_total_initial:,.0f} m3)")

TARGET = 9_000_000
tau, z_alpha = 0.20, 1.65


### 5.10 Calibration EOQ — stock de sécurité (SS)

In [ ]:
D_an_PF = {l: sum(demande_par_pf[l].values()) * (12 / T_max) for l in refs_PF}
D_an_SF = {j: sum(b_bom[j].get(l, 0) * D_an_PF[l] for l in refs_PF) for j in refs_SF}
D_an_MP = {i: sum(a_bom[i].get(j, 0) * D_an_SF[j] for j in refs_SF) for i in refs_MP}

sigma_D_PF = {l: max(1.0, np.std([demande_par_pf[l][t] for t in range(1, T_max+1)])) for l in refs_PF}
sigma_proxy = float(np.mean(list(sigma_D_PF.values())))
sigma_D_MP = {i: sigma_proxy for i in refs_MP}
sigma_D_SF = {j: sigma_proxy for j in refs_SF}

def calc_ss(sigma, delai):
    return z_alpha * sigma * math.sqrt(max(delai, 1))

SS_MP = {i: calc_ss(sigma_D_MP[i], LT[i])   for i in refs_MP}
SS_SF = {j: calc_ss(sigma_D_SF[j], ct_SF[j]) for j in refs_SF}
SS_PF = {l: calc_ss(sigma_D_PF[l], ct_PF[l]) for l in refs_PF}

print("Stocks de sécurité calculés")


### 5.11 Construction du `DataPortal` et de l'instance concrète

In [ ]:
penalite_rupture = 50 * float(np.mean(list(prix_PF.values())))

data = DataPortal()
data['T']         = T_max
data['t_juillet'] = t_juillet
data['AoutSet']   = aout_t

data['MP'] = refs_MP
data['SF'] = refs_SF
data['PF'] = refs_PF

data['TARGET']   = TARGET
data['tau']      = tau
data['z_alpha']  = z_alpha
data['M']        = M_bigM
data['V_max']    = V_max
data['penalite_rupture'] = penalite_rupture

data['prix_MP']       = prix_MP
data['stock_init_MP'] = stock_init_MP
data['LT']            = LT
data['MOQ']           = MOQ
data['volume_MP']     = volume_MP
data['SS_MP']         = SS_MP

data['prix_SF']       = prix_SF
data['stock_init_SF'] = stock_init_SF
data['ct_SF']         = ct_SF
data['MLS_SF']        = MLS_SF
data['charge_SF']     = charge_SF
data['volume_SF']     = volume_SF
data['SS_SF']         = SS_SF

data['prix_PF']       = prix_PF
data['stock_init_PF'] = stock_init_PF
data['ct_PF']         = ct_PF
data['MLS_PF']        = MLS_PF
data['charge_PF']     = charge_PF
data['volume_PF']     = volume_PF
data['SS_PF']         = SS_PF

data['demande'] = demande
data['OC']      = {(i,t): OC[i][t] for i in refs_MP for t in range(1,T_max+1)}
data['WIP_SF']  = {(j,t): WIP_SF[j][t] for j in refs_SF for t in range(1,T_max+1)}
data['WIP_PF']  = {(l,t): WIP_PF[l][t] for l in refs_PF for t in range(1,T_max+1)}

data['a'] = {(i,j): v for i in a_bom for j,v in a_bom[i].items() if v > 0}
data['b'] = {(j,l): v for j in b_bom for l,v in b_bom[j].items() if v > 0}

data['Cap_SF'] = Cap_SF
data['Cap_PF'] = Cap_PF

instance = model.create_instance(data)
print("Instance concrete creee -", len(refs_MP)+len(refs_SF)+len(refs_PF), "references,", T_max, "periodes")


### 5.12 Résolution

In [ ]:
resultats = resoudre(instance, solveur="cbc", verbose=True, temps_max_s=900, gap_mip=0.02)
print("Statut :", resultats.solver.termination_condition)
print("Valeur objectif :", value(instance.cost))


### 5.13 Résultats — valeur de stock, cible, ruptures

In [ ]:
T = instance.T
valeur_finale = (
    sum(instance.prix_MP[i]*value(instance.S_MP[i,T]) for i in instance.MP)
  + sum(instance.prix_SF[j]*value(instance.S_SF[j,T]) for j in instance.SF)
  + sum(instance.prix_PF[l]*value(instance.S_PF[l,T]) for l in instance.PF)
)
total_rupture = sum(value(instance.Rupture_PF[l,t]) for l in instance.PF for t in instance.PER)

print(f"Valeur de stock initiale (calibrée) : {valeur_calibree:,.0f} $")
print(f"Valeur de stock finale (mois {T})    : {valeur_finale:,.0f} $")
print(f"Cible fiscale (TARGET)               : {TARGET:,.0f} $")
print("cible respectee" if valeur_finale <= TARGET else "cible depassee")
print(f"\nRuptures totales (backorder PF)     : {total_rupture:,.0f} unites")
if total_rupture > 0:
    print("  -> concentrees sur les references a lead time tres long (> horizon)")
